# Graph Fraud Detection — Model Monitoring

Sets up drift detection and ML observability for `FRAUD_RING_MODEL`.

| Component | Description |
|-----------|-------------|
| Inference autocapture | Dynamic Table reading `INFERENCE_TABLE()` |
| Monitoring baseline | Training distribution snapshot |
| Model Monitor | PSI drift detection + prediction stat metrics |
| Drift queries | `MODEL_MONITOR_DRIFT_METRIC` on CLAIM_AMOUNT |

In [ ]:
import os, time
import pandas as pd
import snowflake.connector
from snowflake.snowpark import Session

DATABASE      = 'GRAPH_FRAUD_DEMO'
SCHEMA        = 'MODELS'
WAREHOUSE     = 'GRAPH_FRAUD_WH'
MODEL_NAME    = 'FRAUD_RING_MODEL'
MODEL_VERSION = 'V2'
MONITOR_NAME  = 'GRAPH_FRAUD_MONITOR'

# Load PAT from Snowflake config
import tomllib, pathlib
#_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name='parker_demo')
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Connected: {session.get_current_account()}')

## Step 1: Inference Autocapture Dynamic Table

Reads Snowflake's built-in `INFERENCE_TABLE()` log and extracts the 15 graph-fraud
features + prediction. This is the **source** for the Model Monitor, refreshing every 10 minutes.

In [ ]:
FEATURE_COLS = [
    'EFFECTIVE_COMMUNITY_SIZE',
    'DEGREE',
    'SHARED_ADDRESS_COUNT',
    'SHARED_IP_COUNT',
    'AVG_COMMUNITY_REFUND',
    'COMMUNITY_REFUND_STDDEV',
    'ADDR_N_FILERS',
    'ADDR_AVG_REFUND',
    'IP_N_FILERS',
    'IP_AVG_REFUND',
    'SUBMISSION_HOUR',
    'IS_HIGH_CLAIM',
    'IS_ODD_HOUR',
    'CLAIM_AMOUNT',
    'HAS_DIRECT_GRAPH',
]

feature_sql = ',\n    '.join([
    f'RECORD_ATTRIBUTES:"snow.model_serving.request.data.{col}"::FLOAT AS {col}'
    for col in FEATURE_COLS
])

autocapture_sql = f"""
CREATE OR REPLACE DYNAMIC TABLE {DATABASE}.{SCHEMA}.GRAPH_FRAUD_INFERENCE_AUTOCAPTURE
    TARGET_LAG = '10 minutes'
    WAREHOUSE = {WAREHOUSE}
AS
SELECT
    TIMESTAMP::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP,
    RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::FLOAT AS PREDICTION,
    {feature_sql}
FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
"""

session.sql(autocapture_sql).collect()
print('GRAPH_FRAUD_INFERENCE_AUTOCAPTURE created (10-min refresh)')

## Step 2: Monitoring Baseline

A 5,000-row snapshot of the training distribution used as the reference for drift detection.

In [ ]:
# Build monitoring baseline from ACTUAL training data distributions
# Graph FV offline data joined with APPLICATIONS for filing-time features
from snowflake.ml.feature_store import FeatureStore, CreationMode
import pandas as pd

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name='FEATURES',
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

graph_fv = fs.get_feature_view('GRAPH_COMMUNITY_FEATURES', 'V1')
raw_pdf = fs.read_feature_view(graph_fv, store_type='offline').limit(5000).to_pandas()

# Get actual filing-time features from APPLICATIONS for the same applicants
tp_ids = raw_pdf['APPLICANT_ID'].tolist()
tp_list = ','.join(str(t).strip() for t in tp_ids)
filing_pdf = session.sql(f"""
    SELECT
        APPLICANT_ID,
        CLAIM_AMOUNT,
        HOUR(SUBMITTED_AT) AS SUBMISSION_HOUR,
        IFF(CLAIM_AMOUNT > 10000, 1.0, 0.0) AS IS_HIGH_CLAIM,
        IFF(HOUR(SUBMITTED_AT) BETWEEN 0 AND 5, 1.0, 0.0) AS IS_ODD_HOUR
    FROM {DATABASE}.RAW.APPLICATIONS
    WHERE APPLICANT_ID IN ({tp_list})
    QUALIFY ROW_NUMBER() OVER (PARTITION BY APPLICANT_ID ORDER BY SUBMITTED_AT DESC) = 1
""").to_pandas()

# Merge graph FV features with actual filing features
# Align APPLICANT_ID types for merge
raw_pdf['APPLICANT_ID'] = raw_pdf['APPLICANT_ID'].astype(str)
filing_pdf['APPLICANT_ID'] = filing_pdf['APPLICANT_ID'].astype(str)
merged = raw_pdf.merge(filing_pdf, on='APPLICANT_ID', how='left')

baseline_pdf = pd.DataFrame({
    'EFFECTIVE_COMMUNITY_SIZE': merged.get('COMMUNITY_SIZE', pd.Series(dtype=float)).fillna(0),
    'DEGREE': merged.get('DEGREE', pd.Series(dtype=float)).fillna(0),
    'SHARED_ADDRESS_COUNT': merged.get('SHARED_ADDRESS_COUNT', pd.Series(dtype=float)).fillna(0),
    'SHARED_IP_COUNT': merged.get('SHARED_IP_COUNT', pd.Series(dtype=float)).fillna(0),
    'AVG_COMMUNITY_REFUND': merged.get('AVG_COMMUNITY_REFUND', pd.Series(dtype=float)).fillna(0),
    'COMMUNITY_REFUND_STDDEV': merged.get('COMMUNITY_REFUND_STDDEV', pd.Series(dtype=float)).fillna(0),
    'ADDR_N_FILERS': merged.get('ADDR_N_FILERS', pd.Series(dtype=float)).fillna(0),
    'ADDR_AVG_REFUND': merged.get('ADDR_AVG_REFUND', pd.Series(dtype=float)).fillna(0),
    'IP_N_FILERS': merged.get('IP_N_FILERS', pd.Series(dtype=float)).fillna(0),
    'IP_AVG_REFUND': merged.get('IP_AVG_REFUND', pd.Series(dtype=float)).fillna(0),
    'SUBMISSION_HOUR': merged.get('SUBMISSION_HOUR', pd.Series(dtype=float)).fillna(12.0),
    'IS_HIGH_CLAIM': merged.get('IS_HIGH_CLAIM', pd.Series(dtype=float)).fillna(0.0),
    'IS_ODD_HOUR': merged.get('IS_ODD_HOUR', pd.Series(dtype=float)).fillna(0.0),
    'CLAIM_AMOUNT': merged.get('CLAIM_AMOUNT', pd.Series(dtype=float)).fillna(0.0),
    'HAS_DIRECT_GRAPH': 1.0,  # all baseline rows are existing Leiden-processed applicants
}).astype('float64').head(5000)

baseline_pdf['PREDICTION'] = 0.0

session.create_dataframe(baseline_pdf).write.mode('overwrite').save_as_table(
    f'{DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE'
)
print(f'Baseline: {len(baseline_pdf):,} rows with all 15 model features (from actual training data)')

## Step 3: Create Model Monitor

Tracks PSI drift on `CLAIM_AMOUNT` and prediction statistics, refreshing daily.

In [ ]:
session.sql(f"""
CREATE OR REPLACE MODEL MONITOR {DATABASE}.{SCHEMA}.{MONITOR_NAME}
    WITH
    MODEL = {MODEL_NAME}
    VERSION = '{MODEL_VERSION}'
    FUNCTION = 'predict'
    SOURCE = {DATABASE}.{SCHEMA}.GRAPH_FRAUD_INFERENCE_AUTOCAPTURE
    WAREHOUSE = {WAREHOUSE}
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = INFERENCE_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTION')
    BASELINE = {DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE
""").collect()

row = session.sql(
    f"SHOW MODEL MONITORS LIKE '{MONITOR_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}"
).collect()[0].as_dict()
print(f'Monitor  : {MONITOR_NAME}')
print(f'  State  : {row.get("monitor_state", "N/A")}')
print(f'  Task   : {row.get("model_task", "N/A")}')
print(f'  Refresh: {row.get("refresh_interval", "N/A")} | Window: {row.get("aggregation_window", "N/A")}')

## Step 4: Query Drift & Stats

Initially empty — re-run after generating inference traffic with the simulator.

In [ ]:
try:
    drift_rows = session.sql(f"""
        SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            '{MONITOR_NAME}',
            'POPULATION_STABILITY_INDEX',
            'CLAIM_AMOUNT',
            '1 DAY',
            DATEADD('DAY', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
    """).collect()
    if drift_rows:
        print('PSI drift results:')
        for r in drift_rows:
            print(f'  {r.as_dict()}')
    else:
        print('No drift data yet — run simulator/load_test.py to generate inference traffic')
except Exception as e:
    print(f'Drift query: {e}')

try:
    stat_rows = session.sql(f"""
        SELECT * FROM TABLE(MODEL_MONITOR_STAT_METRIC(
            '{MONITOR_NAME}',
            'COUNT',
            'PREDICTION',
            '1 DAY',
            DATEADD('DAY', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
    """).collect()
    if stat_rows:
        print('Prediction count stats:')
        for r in stat_rows:
            print(f'  {r.as_dict()}')
    else:
        print('No stat data yet')
except Exception as e:
    print(f'Stat query: {e}')

In [ ]:
print('=== Model Monitoring Setup Complete ===')
print(f'  Source   : {DATABASE}.{SCHEMA}.GRAPH_FRAUD_INFERENCE_AUTOCAPTURE')
print(f'  Baseline : {DATABASE}.{SCHEMA}.FRAUD_MONITORING_BASELINE')
print(f'  Monitor  : {DATABASE}.{SCHEMA}.{MONITOR_NAME}')
print(f'  Model    : {MODEL_NAME} ({MODEL_VERSION})')
print()
print('View in Snowsight: AI & ML > Models > FRAUD_RING_MODEL > Monitors')
print('Run simulator/load_test.py to generate inference traffic, then re-query drift metrics.')